In [1]:
import pandas as pd

# 1. Find AD, PD and commont proteins
AD_prots = set(pd.read_csv('../networks/AD_nodes.csv')['name'].to_list())
PD_prots = set(pd.read_csv('../networks/PD_nodes.csv')['name'].to_list())
common_prots = AD_prots.intersection(PD_prots)

# 2. Add label colummn
sim_matrix = pd.read_csv('../output/similarity_matrix_all_mean_cleaned.csv')
sim_matrix = sim_matrix.rename(columns={"Unnamed: 0": "name"})
sim_matrix['features'] = sim_matrix.iloc[:, 1:].values.tolist()
sim_matrix = sim_matrix[['name', 'features']]
sim_matrix['label'] = sim_matrix.iloc[:, 0].apply(lambda x: 2 if x in common_prots else (0 if x in AD_prots else 1))
sim_matrix.to_csv('test.csv', index_label='id')

sim_matrix.to_csv('../pre_processing_output/classification_three_labels_nodes_sim.csv', index_label='id')

In [ ]:
# Here I duplicate common proteins to create two disjoint graphs, one for AD and one for PD
# Minor components are considered outliers, thus excluded.

nodes_df = pd.read_csv('../pre_processing_output/duplicated_nodes_main_components.csv')

sim_matrix_dup = sim_matrix.copy()
sim_matrix_dup[sim_matrix_dup['label'] == 2]

df_ad = sim_matrix_dup[sim_matrix_dup['label'] == 2].copy()
df_ad['name'] += '_AD'
df_ad['label'] = 0

df_pd = sim_matrix_dup[sim_matrix_dup['label'] == 2].copy()
df_pd['name'] += '_PD'
df_pd['label'] = 1

sim_matrix_dup = sim_matrix_dup[sim_matrix_dup['label'] != 2]
sim_matrix_dup = pd.concat([sim_matrix_dup, df_ad, df_pd], ignore_index=True)

name_to_sim = dict(zip(sim_matrix_dup['name'], sim_matrix_dup['features']))
nodes_df['GO_embeddings'] = nodes_df['STRING_id'].map(name_to_sim)
nodes_df.to_csv('../pre_processing_output/duplicated_nodes_main_components_sim.csv')